<a href="https://colab.research.google.com/github/dhasan07/-A-weather-monitoring-LSTM/blob/main/A_weather_monitoring_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import files

# --- 1. Load the Kaggle CSV dataset ---
# Check if the zip file is already uploaded in the workspace
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]
if not zip_files:
    print("Please upload your IMD Rainfall dataset zip or CSV file.")
    uploaded = files.upload()
    zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]

# Extract zip if present
csv_filename = None
if zip_files:
    with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
        zip_ref.extractall('.')
        # Find extracted CSV files
        csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
        if csv_files:
            csv_filename = csv_files[0]

if not csv_filename:
    # Fallback to look for raw CSVs in the folder
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv') and f != 'cleaned_normalized_rainfall.csv']
    if csv_files:
        csv_filename = csv_files[0]

if not csv_filename:
    raise FileNotFoundError("No dataset CSV file found. Please ensure you have uploaded the correct archive.")

print(f"Loading dataset: {csv_filename}")
# Correctly parse semicolon-separated CSV values
df_raw = pd.read_csv(csv_filename, sep=';')

# Clean column names (strip quotes)
df_raw.columns = [col.replace('"', '').strip() for col in df_raw.columns]

# Clean string values in state and district
for col in ['state', 'district']:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(str).str.replace('"', '').str.strip()

print("Columns found:", df_raw.columns.tolist())
print(f"Raw dataset shape: {df_raw.shape}")

# --- 2. Convert Daily Records into Chronological Time Series ---
# Sort by state, district, and month to align chronological order
df_raw = df_raw.sort_values(by=['state', 'district', 'month']).reset_index(drop=True)

# Extract day columns ('1st', '2nd', ..., '31st')
day_cols = [f"{i}st" if i==1 else f"{i}nd" if i==2 else f"{i}rd" if i==3 else f"{i}th" for i in range(1, 32)]
# Verify which day columns actually exist
existing_day_cols = [col for col in day_cols if col in df_raw.columns]

# Melt or stack to a single chronological series per district
time_series_list = []
for (state, district), group in df_raw.groupby(['state', 'district'], sort=False):
    for idx, row in group.iterrows():
        month = int(row['month'])
        for d_idx, day_col in enumerate(existing_day_cols):
            day_num = d_idx + 1
            # Handle invalid/non-existent days in specific months (e.g., Feb 30/31)
            if month == 2 and day_num > 29:
                continue
            if month in [4, 6, 9, 11] and day_num > 30:
                continue

            val = row[day_col]
            try:
                val = float(val)
            except (ValueError, TypeError):
                val = np.nan

            time_series_list.append(val)

series_df = pd.DataFrame(time_series_list, columns=['rainfall'])

# Handle missing values & outliers
print(f"Missing values in time-series before cleaning: {series_df['rainfall'].isnull().sum()}")
series_df['rainfall'] = series_df['rainfall'].fillna(method='ffill').fillna(method='bfill')
# Prevent physical impossibility (negative rainfall)
series_df['rainfall'] = series_df['rainfall'].clip(lower=0)

rainfall_data = series_df['rainfall'].values
print(f"Processed continuous chronological sequence length: {len(rainfall_data)}")

# --- 3. Split Chronologically & Normalize ---
split_idx = int(len(rainfall_data) * 0.8)
train_series = rainfall_data[:split_idx].reshape(-1, 1)
test_series = rainfall_data[split_idx:].reshape(-1, 1)

# Scale fitting exclusively on the training set
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_series)
test_scaled = scaler.transform(test_series)

# --- 4. Create Input/Output Sequences (7 days sliding window) ---
def create_sequences(data, seq_length=7):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, 7)
X_test, y_test = create_sequences(test_scaled, 7)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# --- 5. Build and Train LSTM Model ---
model = Sequential([
    LSTM(50, activation='relu', input_shape=(7, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(25, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Training the LSTM Model...")
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# --- 6. Model Evaluation ---
predictions_scaled = model.predict(X_test)
predictions = scaler.inverse_transform(predictions_scaled)
y_test_actual = scaler.inverse_transform(y_test)

# Prevent negative model predictions
predictions = np.clip(predictions, a_min=0, a_max=None)

mae = mean_absolute_error(y_test_actual, predictions)
rmse = np.sqrt(mean_squared_error(y_test_actual, predictions))

print(f"\nTest Metrics:\nMean Absolute Error (MAE): {mae:.4f} mm\nRoot Mean Squared Error (RMSE): {rmse:.4f} mm")

# --- 7. Plotting ---
plt.figure(figsize=(12, 5))

# Subplot 1: Loss Curve
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Training History')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()

# Subplot 2: Actual vs Predicted
plt.subplot(1, 2, 2)
plt.plot(y_test_actual[:100], label='Actual Rainfall', color='blue')
plt.plot(predictions[:100], label='Predicted Rainfall', color='orange', linestyle='--')
plt.title('Actual vs Predicted Rainfall (First 100 Test Samples)')
plt.xlabel('Days')
plt.ylabel('Rainfall (mm)')
plt.legend()
plt.tight_layout()
plt.show()

# --- 8. Real-time Interactive Prediction Dashboard ---
print("\n--- Interactive Weather Prediction System ---")
print("Please input the rainfall values (mm) for the previous 7 days to predict Day 8.")

# Input fields
inputs = [widgets.BoundedFloatText(value=0.0, min=0.0, max=1000.0, step=0.1, description=f"Day {i} (mm):") for i in range(1, 8)]
predict_btn = widgets.Button(description="Predict Next-Day Rainfall", button_style='primary', layout=widgets.Layout(width='250px'))
reset_btn = widgets.Button(description="Reset Inputs", button_style='warning', layout=widgets.Layout(width='150px'))
output_area = widgets.Output()

def handle_prediction(b):
    with output_area:
        clear_output()
        try:
            vals = [inp.value for inp in inputs]
            # Validate inputs are non-negative
            if any(v < 0 for v in vals):
                print("Error: Rainfall inputs cannot be negative.")
                return

            # Scale values using the fitted scaler
            vals_arr = np.array(vals).reshape(-1, 1)
            scaled_vals = scaler.transform(vals_arr)

            # Reshape for LSTM format (1, 7, 1)
            input_seq = scaled_vals.reshape(1, 7, 1)

            # Predict
            pred_scaled = model.predict(input_seq, verbose=0)
            pred_actual = scaler.inverse_transform(pred_scaled)[0][0]

            # Clip negative forecast output
            pred_actual = max(0.0, pred_actual)

            print("\n=== Prediction Results ===")
            print(f"Previous 7-day input series: {', '.join([f'{v:.1f} mm' for v in vals])}")
            print(f"Predicted Day 8 Rainfall: {pred_actual:.2f} mm")
            print("\n*Note: Predictions are modeled estimations based on the training pattern of historical 2022 patterns and not live meteorological observations.")
        except Exception as e:
            print(f"Error processing prediction: {str(e)}")

def handle_reset(b):
    with output_area:
        clear_output()
    for inp in inputs:
        inp.value = 0.0
    print("Inputs reset successfully.")

predict_btn.on_click(handle_prediction)
reset_btn.on_click(handle_reset)

# Render dashboard UI
display(widgets.VBox([
    *inputs,
    widgets.HBox([predict_btn, reset_btn]),
    output_area
]))


In [ ]:
import numpy as np

# Define a sample 7-day input sequence (in mm) to test the model
test_sequence = [2.0, 0.0, 5.0, 8.0, 1.0, 0.0, 4.0]

# Reshape and normalize using the fitted scaler
sequence_arr = np.array(test_sequence).reshape(-1, 1)
scaled_sequence = scaler.transform(sequence_arr)

# Reshape to LSTM format: (1, 7, 1)
input_seq = scaled_sequence.reshape(1, 7, 1)

# Predict using the trained model
predicted_scaled = model.predict(input_seq, verbose=0)
predicted_raw = scaler.inverse_transform(predicted_scaled)[0][0]

# Clip negative predictions to 0.0
predicted_rainfall = max(0.0, predicted_raw)

print("=== Model Test Prediction ===")
print(f"Previous 7-day rainfall: {', '.join(map(str, test_sequence))} mm")
print(f"Predicted Day 8 rainfall: {predicted_rainfall:.2f} mm")


In [ ]:
import os

# Save the model in the recommended native Keras format
model.save('rainfall_lstm_model.keras')
print("Model successfully saved as 'rainfall_lstm_model.keras'")

# Alternatively, you can save it in the legacy HDF5 format
model.save('rainfall_lstm_model.h5')
print("Model successfully saved as 'rainfall_lstm_model.h5'")

# Verify files exist in the workspace
print("\nSaved files in your current directory:")
for f in os.listdir('.'):
    if f.startswith('rainfall_lstm_model'):
        print(f"- {f}")


In [ ]:
import numpy as np
import tensorflow as tf

# 1. Load the saved model
reloaded_model = tf.keras.models.load_model('rainfall_lstm_model.keras')
print("Model successfully reloaded from 'rainfall_lstm_model.keras'")

# 2. Test with a sample sequence
test_seq = [2.0, 0.0, 5.0, 8.0, 1.0, 0.0, 4.0]

# Apply the existing fitted scaler to normalize the inputs
scaled_seq = scaler.transform(np.array(test_seq).reshape(-1, 1))
input_data = scaled_seq.reshape(1, 7, 1)

# 3. Generate a prediction
pred_scaled = reloaded_model.predict(input_data, verbose=0)
pred_val = scaler.inverse_transform(pred_scaled)[0][0]
pred_val = max(0.0, pred_val)

print(f"\nInput Sequence: {test_seq}")
print(f"Reloaded Model Prediction: {pred_val:.2f} mm")


In [ ]:
# Check missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Identify numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Fill missing numeric values with column medians
df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

# Fill missing categorical values with the most frequent value
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols:
    if df[col].isnull().any():
        mode_values = df[col].mode()
        if not mode_values.empty:
            df[col] = df[col].fillna(mode_values.iloc[0])

print("\nMissing values after cleaning:")
print(df.isnull().sum())

In [ ]:
# Find rainfall columns
rainfall_cols = [
    col for col in df.columns
    if "rainfall" in col.lower() or "rain" in col.lower()
    and pd.api.types.is_numeric_dtype(df[col])
]

# Correctly filter to numeric rainfall columns
rainfall_cols = [
    col for col in df.columns
    if ("rainfall" in col.lower() or "rain" in col.lower())
    and pd.api.types.is_numeric_dtype(df[col])
]

if rainfall_cols:
    scaler = MinMaxScaler()

    df[rainfall_cols] = scaler.fit_transform(
        df[rainfall_cols]
    )

    print("Normalized rainfall columns:", rainfall_cols)
else:
    print("No numeric rainfall columns found.")
    print("Check df.columns and select the rainfall columns manually.")

print(df.head())

In [ ]:
df.to_csv("cleaned_normalized_rainfall.csv", index=False)

print("Cleaned and normalized dataset saved successfully.")

files.download("cleaned_normalized_rainfall.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Upload dataset
uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())